# Capstone — Which pages are about to lose search visibility? A leakage-safe decline-risk ranking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Ibtissam Toure · **Lane:** Freestyle — growth / momentum (decline-risk) prediction · **Paper:** https://ibtissamtoure.github.io/FlyRank-Internship/

This notebook is the single source of every number, table and chart in the deployed paper. It runs top to bottom on the starter release in this repo (`data/raw/content_refresh_anonymized.csv`), in under a minute, with seed 42.

> Notebook ↔ paper map: §1 Question → Introduction · §2 Data → Data · §3 Methodology → Methodology · §4 Results → Results · §5 Limitations → Limitations · §6 Recommendations → Ranked recommendations · §7 Artifacts → the figures the page embeds · §8 ML-12 → demo / social / employer cuts.

## 1. Question

**Research question.** Using only what is known about a page *30 days before* the end of the window, can we rank pages so that the ones about to lose ≥ 20% of their search impressions in the next 30 days come first — better than the simple rules an editor would otherwise use?

**The decision it supports.** A content team cannot review every page each month. They need a short, per-client review list: *which 20 pages should a person look at first?* A false positive costs one wasted review; a false negative means a page that was about to lose visibility is noticed a month late.

**Where this came from.** Weeks 1–3 framed a *future* decline task and showed (W03) that `trend_pct` leaks the label. Weeks 5–7 then trained on the snapshot label with 90-day features that **overlap the label window**. The capstone fixes that: it rebuilds the task so every feature comes strictly from *before* the label window — the clean future-window label the freestyle brief asks for.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

current = Path.cwd()
project_root = next(
    (p for p in [current, *current.parents]
     if (p / "data/raw/content_refresh_anonymized.csv").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

OUT = project_root / "work" / "outputs"
IMG = project_root / "docs" / "img"
OUT.mkdir(parents=True, exist_ok=True)
IMG.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(project_root / "data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(raw):,} | columns: {raw.shape[1]} | clients: {raw['client_id'].nunique()}")

Rows: 30,000 | columns: 44 | clients: 32


## 2. Data

**Release.** FlyRank ML Internship dataset — the 30,000-page anonymized starter slice shipped in this repo (32 pseudonymized clients, trailing 90-day window). The warehouse release on Hugging Face was used in Week 3 for the data contract (March 2026 partition, 9.84M daily rows, grain verified unique); the capstone model runs on the starter slice because it already carries the three non-overlapping 30-day windows the design needs.

**The three windows hidden in the 90-day columns.** The slice has 90-day totals plus *last 30 days* and *previous 30 days* columns. Subtracting gives the oldest 30 days:

| Window | Days back from export | How it is obtained | Role |
|---|---|---|---|
| W3 | 61–90 | `*_90d − *_last_30d − *_prev_30d` | feature |
| W2 | 31–60 | `*_prev_30d` | feature |
| — | **T = day 30** | prediction moment | — |
| W1 | 1–30 | `*_last_30d` | **label only** |

**Exclusions (and why).**
- Pages with fewer than 10 impressions in W2 (6,273 pages): a ±20% change on single-digit counts is noise, and pages with 0 impressions in W2 cannot "decline".
- Any column aggregated over the full 90 days (`impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `days_with_*`, tiers built on them): they contain W1, the label window.
- `days_since_last_update`: 68% of pages were last updated *inside* W1, so its value at T is unknown.
- `trend_direction`, `trend_pct`: they *are* the label.
- `provider_used`, `model_used`: the data dictionary marks them "not a model feature".
- `content_id`, `client_id`: pseudonymous IDs — used only for grouping the split, never as features.

In [2]:
df = raw.copy()
for m in ["impressions", "clicks", "sessions"]:
    df[f"{m}_61_90d"] = df[f"{m}_90d"] - df[f"{m}_last_30d"] - df[f"{m}_prev_30d"]
    assert (df[f"{m}_61_90d"] >= 0).all(), f"negative derived window for {m}"

# Label rule reconstructed from the data dictionary, checked against trend_direction
recon = (df["impressions_prev_30d"] > 0) & (df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"])
assert (recon == df["trend_direction"].eq("down")).all()
print("Label rule reproduces trend_direction == 'down' on all", len(df), "rows")

updated_in_label_window = (df["days_since_last_update"] <= 30).mean()
print(f"Pages last updated inside the label window (<=30 days): {updated_in_label_window:.1%}")

MIN_W2_IMPRESSIONS = 10
data = df[df["impressions_prev_30d"] >= MIN_W2_IMPRESSIONS].copy().reset_index(drop=True)
y = (data["impressions_last_30d"] < 0.8 * data["impressions_prev_30d"]).astype(int).values
groups = data["client_id"].values

data_summary = {
    "starter_rows": int(len(raw)),
    "eligible_pages": int(len(data)),
    "excluded_low_volume": int(len(raw) - len(data)),
    "clients": int(data["client_id"].nunique()),
    "base_rate_decline": round(float(y.mean()), 4),
}
data_summary

Label rule reproduces trend_direction == 'down' on all 30000 rows
Pages last updated inside the label window (<=30 days): 68.3%


{'starter_rows': 30000,
 'eligible_pages': 23727,
 'excluded_low_volume': 6273,
 'clients': 31,
 'base_rate_decline': 0.6162}

In [3]:
client_sizes = data.groupby("client_id").size().sort_values(ascending=False)
client_base = data.assign(y=y).groupby("client_id")["y"].mean()
print("Pages per client — largest share:", f"{client_sizes.iloc[0] / len(data):.1%}")
print("Clients with >= 100 eligible pages:", int((client_sizes >= 100).sum()))
print("Per-client decline base rate: min {:.2f} | median {:.2f} | max {:.2f}".format(
    client_base.min(), client_base.median(), client_base.max()))

Pages per client — largest share: 28.9%
Clients with >= 100 eligible pages: 20
Per-client decline base rate: min 0.00 | median 0.64 | max 1.00


## 3. Methodology

**Assumptions.** (1) The 30-day windows are aligned calendar blocks ending at the same export date for every page. (2) Page metadata (content type, intent, keyword context, word count) did not change between T and export. (3) Content age at T = age at export − 30 days.

**Label.** `decline = 1` when impressions in W1 are more than 20% below W2 (the pipeline's own "down" rule), computed for pages with ≥ 10 W2 impressions.

**Features (all knowable at T).** Log volumes for impressions, clicks and sessions in W2 and W3; momentum W2 vs W3 (log ratio) for each; CTR in W2; content age at T; keyword search volume, competition, CPC; word count; missing-data flags (`has_keyword_data`, `has_word_count` — missingness follows content type, so it is flagged, not zero-filled); content type and main intent (one-hot).

**Baselines (two transparent rules).**
- *Visibility rule* — the Week-4 rule made leakage-safe: rank by W2 impressions (high-visibility pages first).
- *Already-falling rule* — the rule written in Week 2 ("flag pages whose recent impressions dropped"): rank by the W3 → W2 drop.

**Models.** Logistic regression (primary — interpretable, carried over from Week 5) and histogram gradient boosting (a non-linear check). Both fixed before looking at test folds; seed 42.

**Validation.** 5-fold `GroupKFold` by client: every page is scored by a model that never saw *any* page of its client. Metrics: ROC-AUC, average precision, Precision@50 and Precision@200 inside each test fold, **always next to the fold's base rate**. Because a real editor works one client at a time, the operational metric is *within-client* AUC and Precision@20 for the 20 clients with ≥ 100 eligible pages.

**Leakage checks.** (a) Forbidden-column assertion on the feature list; (b) deliberate-leak experiments: add the 90-day aggregates, then `trend_pct`, and watch the score jump.

In [4]:
def build_features(d):
    F = pd.DataFrame(index=d.index)
    for m in ["impressions", "clicks", "sessions"]:
        F[f"log_{m}_w2"] = np.log1p(d[f"{m}_prev_30d"])
        F[f"log_{m}_w3"] = np.log1p(d[f"{m}_61_90d"])
        F[f"momentum_{m}"] = F[f"log_{m}_w2"] - F[f"log_{m}_w3"]
    F["ctr_w2"] = d["clicks_prev_30d"] / d["impressions_prev_30d"] * 100
    F["age_at_T"] = d["content_age_days"] - 30
    F["log_search_volume"] = np.log1p(d["search_volume"])
    F["competition"] = d["competition"]
    F["cpc"] = d["cpc"]
    F["log_word_count"] = np.log1p(d["word_count"])
    F["has_keyword_data"] = d["search_volume"].notna().astype(int)
    F["has_word_count"] = d["word_count"].notna().astype(int)
    F["content_type"] = d["content_type"].fillna("unknown")
    F["main_intent"] = d["main_intent"].fillna("unknown")
    return F

X = build_features(data)
CATEGORICAL = ["content_type", "main_intent"]
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]

FORBIDDEN = {
    "trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "days_with_impressions", "days_with_sessions", "impression_tier", "position_tier",
    "days_since_last_update", "freshness_tier", "content_id", "client_id", "provider_used", "model_used",
}
assert not FORBIDDEN & set(X.columns), FORBIDDEN & set(X.columns)
print(f"{len(X.columns)} features, none forbidden:")
print(", ".join(X.columns))

19 features, none forbidden:
log_impressions_w2, log_impressions_w3, momentum_impressions, log_clicks_w2, log_clicks_w3, momentum_clicks, log_sessions_w2, log_sessions_w3, momentum_sessions, ctr_w2, age_at_T, log_search_volume, competition, cpc, log_word_count, has_keyword_data, has_word_count, content_type, main_intent


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_lr(numeric, categorical):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])

def make_hgb(numeric, categorical):
    pre = ColumnTransformer([
        ("num", "passthrough", numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                         min_samples_leaf=50, random_state=SEED)
    return Pipeline([("pre", pre), ("clf", clf)])

FOLDS = list(GroupKFold(n_splits=5).split(X, y, groups))
for i, (tr, te) in enumerate(FOLDS, 1):
    assert not set(groups[tr]) & set(groups[te])
    print(f"fold {i}: test pages {len(te):>5,} | test clients {len(set(groups[te])):>2} | base rate {y[te].mean():.3f}")

def out_of_fold(make_model, Xf, numeric, categorical):
    scores = np.zeros(len(Xf))
    for tr, te in FOLDS:
        model = make_model(numeric, categorical)
        model.fit(Xf.iloc[tr], y[tr])
        scores[te] = model.predict_proba(Xf.iloc[te])[:, 1]
    return scores

SCORES = {
    "Visibility rule": X["log_impressions_w2"].values,
    "Already-falling rule": -X["momentum_impressions"].values,
    "Logistic regression": out_of_fold(make_lr, X, NUMERIC, CATEGORICAL),
    "Gradient boosting": out_of_fold(make_hgb, X, NUMERIC, CATEGORICAL),
}
print("Out-of-fold scores ready for:", list(SCORES))

fold 1: test pages 6,857 | test clients  1 | base rate 0.494
fold 2: test pages 4,219 | test clients  5 | base rate 0.653
fold 3: test pages 4,218 | test clients  8 | base rate 0.511
fold 4: test pages 4,220 | test clients  8 | base rate 0.771
fold 5: test pages 4,213 | test clients  9 | base rate 0.729


Out-of-fold scores ready for: ['Visibility rule', 'Already-falling rule', 'Logistic regression', 'Gradient boosting']


## 4. Results (vs baseline)

Same pages, same folds, same metrics for every method. Precision@K is read against the base rate in the same fold — a high precision can simply be a high base rate.

In [6]:
def precision_at_k(score, target, k):
    top = np.argsort(-score, kind="mergesort")[:k]
    return float(target[top].mean())

rows = []
for name, s in SCORES.items():
    auc = [roc_auc_score(y[te], s[te]) for _, te in FOLDS]
    ap = [average_precision_score(y[te], s[te]) for _, te in FOLDS]
    p50 = [precision_at_k(s[te], y[te], 50) for _, te in FOLDS]
    p200 = [precision_at_k(s[te], y[te], 200) for _, te in FOLDS]
    rows.append({"method": name,
                 "ROC-AUC": np.mean(auc), "AUC sd": np.std(auc),
                 "Avg precision": np.mean(ap),
                 "Precision@50": np.mean(p50), "Precision@200": np.mean(p200)})
base_rate_folds = float(np.mean([y[te].mean() for _, te in FOLDS]))
rows.append({"method": "Base rate (no ranking)", "ROC-AUC": 0.5, "AUC sd": 0.0,
             "Avg precision": base_rate_folds, "Precision@50": base_rate_folds, "Precision@200": base_rate_folds})
fold_table = pd.DataFrame(rows).set_index("method")
fold_table.round(3)

,ROC-AUC,AUC sd,Avg precision,Precision@50,Precision@200
method,,,,,
Visibility rule,0.505,0.055,0.624,0.504,0.539
Already-falling rule,0.533,0.023,0.647,0.588,0.612
Logistic regression,0.597,0.019,0.712,0.836,0.812
Gradient boosting,0.586,0.031,0.698,0.732,0.766
Base rate (no ranking),0.500,0.000,0.631,0.631,0.631


**Within-client view (how an editor would use it).** For each of the 20 clients with ≥ 100 eligible pages, rank that client's pages and check the top 20.

In [7]:
MIN_CLIENT_PAGES = 100
scored = pd.DataFrame({"client": groups, "y": y, **SCORES})
per_client = []
for client, d in scored.groupby("client"):
    if len(d) < MIN_CLIENT_PAGES or d["y"].nunique() < 2:
        continue
    r = {"client": f"client {len(per_client) + 1:02d}", "pages": len(d), "base_rate": d["y"].mean()}
    for name in SCORES:
        r[f"AUC | {name}"] = roc_auc_score(d["y"], d[name])
        r[f"P@20 | {name}"] = d.nlargest(20, name)["y"].mean()
    per_client.append(r)
per_client = pd.DataFrame(per_client)

summary = {"median base rate": per_client["base_rate"].median()}
for name in SCORES:
    summary[f"median AUC | {name}"] = per_client[f"AUC | {name}"].median()
    summary[f"median P@20 | {name}"] = per_client[f"P@20 | {name}"].median()
wins_vs_rule = int((per_client["AUC | Logistic regression"] > per_client["AUC | Already-falling rule"]).sum())
wins_vs_vis = int((per_client["AUC | Logistic regression"] > per_client["AUC | Visibility rule"]).sum())
lr_above_chance = int((per_client["AUC | Logistic regression"] > 0.5).sum())
print(f"Clients evaluated: {len(per_client)} (labels anonymised as client 01..)")
print(f"LR within-client AUC beats the already-falling rule in {wins_vs_rule}/{len(per_client)} clients,"
      f" the visibility rule in {wins_vs_vis}/{len(per_client)}, and is above 0.5 in {lr_above_chance}/{len(per_client)}")
pd.Series(summary).round(3)

Clients evaluated: 20 (labels anonymised as client 01..)
LR within-client AUC beats the already-falling rule in 16/20 clients, the visibility rule in 18/20, and is above 0.5 in 19/20


median base rate                      0.667
median AUC | Visibility rule          0.454
median P@20 | Visibility rule         0.550
median AUC | Already-falling rule     0.530
median P@20 | Already-falling rule    0.600
median AUC | Logistic regression      0.603
median P@20 | Logistic regression     0.800
median AUC | Gradient boosting        0.577
median P@20 | Gradient boosting       0.725
dtype: float64

**Continuity check — my Week-5 model under the honest label.** Week 5 used snapshot metadata including `days_since_last_update`, `age_tier`, `model_used`. Re-scored on the same folds and the same forward label:

In [8]:
w5 = data.copy()
W5_NUM = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "days_since_last_update"]
W5_CAT = ["competition_level", "content_type", "main_intent", "provider_used", "model_used",
          "age_tier", "freshness_tier", "word_count_tier", "char_count_tier"]
w5[W5_CAT] = w5[W5_CAT].fillna("unknown")
w5_scores = out_of_fold(make_lr, w5[W5_NUM + W5_CAT], W5_NUM, W5_CAT)
w5_auc = float(np.mean([roc_auc_score(y[te], w5_scores[te]) for _, te in FOLDS]))
w5_p50 = float(np.mean([precision_at_k(w5_scores[te], y[te], 50) for _, te in FOLDS]))
print(f"Week-5 feature set (forward label): AUC {w5_auc:.3f} | Precision@50 {w5_p50:.3f}")
print(f"Capstone feature set:               AUC {fold_table.loc['Logistic regression','ROC-AUC']:.3f} | "
      f"Precision@50 {fold_table.loc['Logistic regression','Precision@50']:.3f}")

Week-5 feature set (forward label): AUC 0.552 | Precision@50 0.756
Capstone feature set:               AUC 0.597 | Precision@50 0.836


**Leakage experiments.** What happens if the forbidden columns are let back in?

In [9]:
def lr_auc(extra_numeric):
    Xl = X.copy()
    for c in extra_numeric:
        Xl[c] = data[c].fillna(0)
    s = out_of_fold(make_lr, Xl, NUMERIC + extra_numeric, CATEGORICAL)
    return float(np.mean([roc_auc_score(y[te], s[te]) for _, te in FOLDS]))

AGG90 = ["impressions_90d", "clicks_90d", "sessions_90d", "ctr", "avg_position", "days_with_impressions"]
leak_table = pd.Series({
    "Honest features (knowable at T)": fold_table.loc["Logistic regression", "ROC-AUC"],
    "+ 90-day aggregates (overlap label window)": lr_auc(AGG90),
    "+ trend_pct (the label itself)": lr_auc(["trend_pct"]),
}, name="ROC-AUC (5-fold grouped)")
leak_table.round(3)

Honest features (knowable at T)               0.597
+ 90-day aggregates (overlap label window)    0.687
+ trend_pct (the label itself)                1.000
Name: ROC-AUC (5-fold grouped), dtype: float64

**Robustness.** (a) Week-5's single 80/20 client split (seed 42); (b) a stricter volume floor of 100 W2 impressions.

In [10]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, groups))
lr = make_lr(NUMERIC, CATEGORICAL).fit(X.iloc[tr], y[tr])
s_single = lr.predict_proba(X.iloc[te])[:, 1]
single = {
    "test clients": len(set(groups[te])), "test pages": len(te), "base rate": y[te].mean(),
    "LR AUC": roc_auc_score(y[te], s_single), "LR P@50": precision_at_k(s_single, y[te], 50),
    "Already-falling P@50": precision_at_k(-X["momentum_impressions"].values[te], y[te], 50),
}
mask = data["impressions_prev_30d"].values >= 100
strict_auc = float(roc_auc_score(y[mask], SCORES["Logistic regression"][mask]))
strict_rule = float(roc_auc_score(y[mask], SCORES["Already-falling rule"][mask]))
print("Single 80/20 client split:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in single.items()})
print(f"Floor >= 100 W2 impressions ({mask.sum():,} pages): pooled OOF AUC LR {strict_auc:.3f} vs rule {strict_rule:.3f}")

Single 80/20 client split: {'test clients': 7, 'test pages': 4252, 'base rate': np.float64(0.611), 'LR AUC': 0.628, 'LR P@50': 0.92, 'Already-falling P@50': 0.54}
Floor >= 100 W2 impressions (18,010 pages): pooled OOF AUC LR 0.628 vs rule 0.567


**What the model leans on.** Logistic-regression coefficients on standardized features, fitted on all eligible pages (for interpretation only — every score used for evaluation and the queue is out-of-fold). Positive = associated with higher decline risk. These are associations in one window, not causes.

In [11]:
lr_full = make_lr(NUMERIC, CATEGORICAL).fit(X, y)
names = lr_full.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(lr_full.named_steps["clf"].coef_[0], index=[n.split("__", 1)[1] for n in names])
coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index)
coefs.head(12).round(3)

main_intent_unknown                0.586
content_type_feedly article        0.359
age_at_T                          -0.352
log_clicks_w2                     -0.278
log_clicks_w3                     -0.247
content_type_keyword article       0.229
content_type_comparison article   -0.222
log_impressions_w2                 0.207
log_impressions_w3                 0.182
has_word_count                     0.141
log_search_volume                 -0.141
main_intent_informational         -0.127
dtype: float64

## 5. Limitations

- **One snapshot, one forward month.** The label is a pseudo-forward split of a single export: the model is evaluated on one 30-day outcome window shared by every page. Anything that hit all sites that month (a core update, seasonality) is baked into the base rate (61.6% declined). A rolling multi-month backtest on the warehouse daily table is required before calling this a forecast.
- **Modest discrimination.** AUC ≈ 0.60 means the ranking is clearly better than chance and than both rules, but far from a sharp classifier. It is a prioritisation aid, not a verdict on any single page.
- **Unbalanced panel.** One client holds 29% of eligible pages, and among the 20 clients with ≥ 100 eligible pages the decline base rate ranges from 0.18 to 0.96. Grouped folds and the within-client view keep this honest, but per-client results for small clients are noisy.
- **Metadata assumed stable.** Content type, intent, keyword fields and word count are read at export; if they changed inside the label window, a little leakage remains (no evidence of this, but it cannot be verified from the slice).
- **Associations, not causes.** Nothing here shows that changing a page changes its visibility, and nothing here models or predicts Google's ranking system.
- **Scores are not calibrated probabilities** and the 20% threshold is the pipeline's convention, not a business-validated cost trade-off.

In [12]:
limitations_numbers = {
    "largest_client_share": round(float(client_sizes.iloc[0] / len(data)), 3),
    "client_base_rate_min_100plus": round(float(client_base[client_sizes[client_sizes >= 100].index].min()), 3),
    "client_base_rate_max_100plus": round(float(client_base[client_sizes[client_sizes >= 100].index].max()), 3),
    "updated_inside_label_window": round(float(updated_in_label_window), 3),
}
limitations_numbers

{'largest_client_share': 0.289,
 'client_base_rate_min_100plus': 0.181,
 'client_base_rate_max_100plus': 0.961,
 'updated_inside_label_window': 0.683}

## 6. Ranked recommendations

Every eligible page gets its **out-of-fold** risk score (from a model that never saw its client), ranked *within its client*. Transparent reason codes come from the features themselves; the action follows from risk + visibility. The observed decline rate of each action bucket is reported so a reader can see the evidence behind each tier.

In [13]:
queue = data[["content_id", "client_id", "impressions_prev_30d", "clicks_prev_30d"]].copy()
queue["risk_score"] = SCORES["Logistic regression"]
queue["client_rank"] = queue.groupby("client_id")["risk_score"].rank(ascending=False, method="first").astype(int)
queue["client_pct"] = queue.groupby("client_id")["risk_score"].rank(pct=True)
queue["declined_next_30d"] = y

def reasons(i):
    r = []
    if X.at[i, "age_at_T"] < 180: r.append("young_page")
    if data.at[i, "clicks_prev_30d"] == 0: r.append("no_clicks_w2")
    if X.at[i, "momentum_impressions"] < np.log(0.8): r.append("already_falling")
    if X.at[i, "momentum_impressions"] > np.log(1.25): r.append("recent_spike")
    if data.at[i, "impressions_prev_30d"] >= 3000: r.append("high_visibility")
    if X.at[i, "has_keyword_data"] == 0 or X.at[i, "main_intent"] == "unknown": r.append("thin_metadata")
    return ", ".join(r) if r else "none"

queue["reason_codes"] = [reasons(i) for i in queue.index]

def action(row):
    top = row["client_pct"] >= 0.90
    watch = row["client_pct"] >= 0.75
    if top and row["impressions_prev_30d"] >= 1000:
        return "1 Protect: review now"
    if top and row["clicks_prev_30d"] == 0:
        return "2 Improve: snippet / intent review"
    if top:
        return "3 Investigate: refresh or merge candidate"
    if watch:
        return "4 Monitor: re-check next month"
    return "5 No action"

queue["action"] = queue.apply(action, axis=1)
playbook = (queue.groupby("action")
            .agg(pages=("content_id", "size"),
                 observed_decline_rate=("declined_next_30d", "mean"),
                 median_w2_impressions=("impressions_prev_30d", "median"))
            .reset_index())
playbook["lift_vs_base"] = playbook["observed_decline_rate"] / y.mean()
queue = queue.sort_values(["action", "client_id", "client_rank"]).reset_index(drop=True)
queue.drop(columns=["declined_next_30d"]).to_csv(OUT / "capstone_action_queue.csv", index=False)
print("Queue written (gitignored):", OUT / "capstone_action_queue.csv")
playbook.round(3)

Queue written (gitignored): /root/repo/work/outputs/capstone_action_queue.csv


,action,pages,observed_decline_rate,median_w2_impressions,lift_vs_base
0,1 Protect: review now,750,0.787,1976.5,1.277
1,2 Improve: snippet / intent review,1476,0.736,330.0,1.195
2,3 Investigate: refresh or merge candidate,163,0.724,550.0,1.175
3,4 Monitor: re-check next month,3561,0.698,360.0,1.133
4,5 No action,17777,0.582,414.0,0.944


In [14]:
reason_rates = []
for code_ in ["young_page", "no_clicks_w2", "already_falling", "recent_spike", "high_visibility", "thin_metadata"]:
    has = queue["reason_codes"].str.contains(code_)
    reason_rates.append({"reason_code": code_, "pages": int(has.sum()),
                         "decline_rate_with": queue.loc[has, "declined_next_30d"].mean(),
                         "decline_rate_without": queue.loc[~has, "declined_next_30d"].mean()})
reason_rates = pd.DataFrame(reason_rates)
reason_rates.round(3)

,reason_code,pages,decline_rate_with,decline_rate_without
0,young_page,10091,0.716,0.542
1,no_clicks_w2,12093,0.648,0.583
2,already_falling,12528,0.653,0.575
3,recent_spike,5239,0.597,0.622
4,high_visibility,3852,0.560,0.627
5,thin_metadata,872,0.658,0.615


In [15]:
example = (queue[queue["action"].str.startswith("1")]
           .sort_values("risk_score", ascending=False)
           .head(10)[["content_id", "client_rank", "impressions_prev_30d", "risk_score", "reason_codes", "action"]])
example.round(3)

,content_id,client_rank,impressions_prev_30d,risk_score,reason_codes,action
410,content_d4786a3f7544,1,2775,0.949,"no_clicks_w2, already_falling, thin_metadata",1 Protect: review now
411,content_6e28a04c07a8,2,17445,0.945,"young_page, no_clicks_w2, recent_spike, high_v...",1 Protect: review now
412,content_a247759cb944,5,4585,0.937,"young_page, no_clicks_w2, high_visibility",1 Protect: review now
413,content_f988b4cba4ea,6,5112,0.937,"young_page, no_clicks_w2, already_falling, hig...",1 Protect: review now
414,content_b3b2e73f3491,9,1593,0.934,"young_page, no_clicks_w2",1 Protect: review now
415,content_70dba941a4cd,10,1572,0.931,"young_page, no_clicks_w2",1 Protect: review now
416,content_81a16ef4d700,12,2551,0.930,"young_page, no_clicks_w2, already_falling",1 Protect: review now
417,content_873decd76dc4,13,2690,0.929,"young_page, no_clicks_w2",1 Protect: review now
418,content_39b6f6812582,15,12138,0.929,"young_page, high_visibility",1 Protect: review now
419,content_8b0cad0716e6,16,1725,0.928,"young_page, no_clicks_w2",1 Protect: review now


## 7. Artifacts the paper embeds

Four figures (saved to `docs/img/`) and one metrics receipt (`work/outputs/capstone_metrics.json`, committed). Colours: model = blue, already-falling rule = orange, gradient boosting = aqua, visibility rule / base rate = grey; every series is also direct-labelled.

In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
COL = {"Logistic regression": "#2a78d6", "Already-falling rule": "#eb6834",
       "Gradient boosting": "#1baf7a", "Visibility rule": "#8a8984"}
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                     "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "svg.fonttype": "none"})

# Figure 1 — Precision@K curve inside grouped test folds
ks = np.arange(10, 501, 10)
fig, ax = plt.subplots(figsize=(8, 4.6))
for name in ["Logistic regression", "Gradient boosting", "Already-falling rule", "Visibility rule"]:
    curve = [np.mean([precision_at_k(SCORES[name][te], y[te], k) for _, te in FOLDS]) for k in ks]
    ax.plot(ks, curve, lw=2, color=COL[name])
    dy = {"Logistic regression": 6, "Gradient boosting": -6}.get(name, 0)
    ax.annotate(name, (ks[-1], curve[-1]), xytext=(6, dy), textcoords="offset points",
                va="center", color=INK2, fontsize=10)
ax.axhline(base_rate_folds, color=INK2, lw=1.2, ls=(0, (4, 3)))
ax.annotate(f"Base rate {base_rate_folds:.0%}", (12, base_rate_folds), xytext=(0, 6),
            textcoords="offset points", color=INK2, fontsize=10)
ax.set_xlabel("Pages reviewed per test fold (K)")
ax.set_ylabel("Share that declined ≥20% next month")
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlim(0, 640); ax.set_ylim(0.4, 0.95)
fig.tight_layout(); fig.savefig(IMG / "fig1_precision_at_k.svg"); plt.close(fig)

# Figure 2 — within-client AUC, one dot per client
order = per_client.sort_values("AUC | Logistic regression")
fig, ax = plt.subplots(figsize=(8, 6))
ypos = np.arange(len(order))
for name, off in [("Visibility rule", -0.2), ("Already-falling rule", 0.0), ("Logistic regression", 0.2)]:
    ax.scatter(order[f"AUC | {name}"], ypos + off, s=46, color=COL[name], edgecolor=SURF, linewidth=1.5,
               label=name, zorder=3)
ax.axvline(0.5, color=INK2, lw=1.2, ls=(0, (4, 3)))
ax.set_yticks(ypos); ax.set_yticklabels([f"{c} (n={n})" for c, n in zip(order["client"], order["pages"])], fontsize=9)
ax.set_xlabel("Within-client ROC-AUC (0.5 = random ordering)")
ax.legend(loc="lower right", frameon=False, fontsize=10)
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(IMG / "fig2_within_client_auc.svg"); plt.close(fig)

# Figure 3 — leakage experiment
fig, ax = plt.subplots(figsize=(8, 2.8))
labels = list(leak_table.index)[::-1]; vals = list(leak_table.values)[::-1]
colors = ["#8a8984", "#8a8984", "#2a78d6"]
ax.barh(labels, vals, color=colors, height=0.5)
for i, v in enumerate(vals):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", color=INK, fontsize=10)
ax.set_xlim(0.5, 1.08); ax.set_xlabel("ROC-AUC, 5-fold grouped by client")
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(IMG / "fig3_leakage.svg"); plt.close(fig)

# Figure 4 — observed decline rate per action bucket
pb = playbook.sort_values("action")
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.barh(pb["action"][::-1], pb["observed_decline_rate"][::-1], color="#2a78d6", height=0.55)
for i, (v, n) in enumerate(zip(pb["observed_decline_rate"][::-1], pb["pages"][::-1])):
    ax.text(v + 0.012, i, f"{v:.0%}  ({n:,} pages)", va="center", color=INK, fontsize=10,
            bbox=dict(facecolor=SURF, edgecolor="none", pad=1.5))
ax.axvline(y.mean(), color=INK2, lw=1.2, ls=(0, (4, 3)))
ax.text(y.mean() + 0.005, len(pb) - 0.45, f"overall {y.mean():.0%}", color=INK2, fontsize=9)
ax.set_xlim(0, 1.15); ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlabel("Observed share that declined ≥20% next month (out-of-fold)")
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(IMG / "fig4_action_buckets.svg"); plt.close(fig)
print(sorted(p.name for p in IMG.glob("*.svg")))

['fig1_precision_at_k.svg', 'fig2_within_client_auc.svg', 'fig3_leakage.svg', 'fig4_action_buckets.svg']


In [17]:
def r(x):
    return round(float(x), 4)

metrics = {
    "seed": SEED,
    "data": data_summary,
    "validation": "GroupKFold(n_splits=5) by client_id; out-of-fold scores",
    "label": "impressions_last_30d < 0.8 * impressions_prev_30d, pages with impressions_prev_30d >= 10",
    "features": list(X.columns),
    "fold_metrics": {m: {k: r(v) for k, v in row.items()} for m, row in fold_table.iterrows()},
    "within_client": {"clients": int(len(per_client)), "min_pages": MIN_CLIENT_PAGES,
                      **{k: r(v) for k, v in summary.items()},
                      "lr_auc_beats_already_falling_rule": wins_vs_rule,
                      "lr_auc_beats_visibility_rule": wins_vs_vis,
                      "lr_auc_above_chance": lr_above_chance},
    "week5_features_forward_label": {"auc": r(w5_auc), "precision_at_50": r(w5_p50)},
    "leakage_experiment_auc": {k: r(v) for k, v in leak_table.items()},
    "single_group_split_seed42": {k: (r(v) if isinstance(v, float) else int(v)) for k, v in single.items()},
    "strict_floor_100": {"pages": int(mask.sum()), "lr_auc": r(strict_auc), "rule_auc": r(strict_rule)},
    "playbook": [{k: (r(v) if isinstance(v, (float, np.floating)) else (int(v) if isinstance(v, (np.integer,)) else v))
                  for k, v in row.items()} for row in playbook.to_dict("records")],
    "reason_codes": [{k: (r(v) if isinstance(v, (float, np.floating)) else (int(v) if isinstance(v, (np.integer,)) else v))
                      for k, v in row.items()} for row in reason_rates.to_dict("records")],
    "limitations": limitations_numbers,
    "top_coefficients": {k: r(v) for k, v in coefs.head(12).items()},
}
(OUT / "capstone_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved", OUT / "capstone_metrics.json")

Saved /root/repo/work/outputs/capstone_metrics.json


## 8. ML-12 — demo outline, social post, employer summary

### 5-minute demo outline
1. **(0:00–0:40) The problem.** A content team can review ~20 pages per client per month. Which 20?
2. **(0:40–1:40) The trap I found.** My Week-5 model used 90-day features that overlap the label month; `trend_pct` alone gives AUC ≈ 1.0. Show Figure 3.
3. **(1:40–2:40) The fix.** Split the 90 days into three 30-day windows: learn from days 31–90, predict days 1–30. Show the window table.
4. **(2:40–3:50) The result.** Out-of-fold, grouped by client: AUC 0.60 vs 0.53 / 0.51 for the two rules; top-50 precision 84% vs a 63% base rate; better than the rule inside 16 of 20 clients. Show Figures 1–2.
5. **(3:50–4:40) The playbook.** Protect / Improve / Investigate / Monitor, with reason codes and the observed decline rate per bucket (Figure 4).
6. **(4:40–5:00) Limits + next step.** One forward month; next is a rolling backtest on the warehouse with June 2026 sealed.

### Social-post cut
> Which pages are about to lose search visibility? On 23.7k real (anonymized) pages from the FlyRank dataset, I learned from days 31–90 and predicted the next 30 — validated on clients the model never saw. The top-50 review list caught declines 84% of the time vs a 63% base rate, and beat "flag what's already falling" (a rule that is barely better than random here). Bonus: one leaked column would have given me a fake perfect score. Paper → https://ibtissamtoure.github.io/FlyRank-Internship/

### Employer-facing summary (3 sentences)
I built a leakage-safe decline-risk ranking for search content on 23,727 real, anonymized pages from 31 client sites in the FlyRank ML Internship dataset. Using only signals available 30 days before the outcome and validating on held-out clients, the model ranks next-month visibility losses better than both editor rules (AUC 0.60 vs 0.53 and 0.51; top-50 precision 84% vs a 63% base rate). I turned it into a reason-coded review playbook and documented its limits openly in a deployed research paper.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (pseudonymous IDs only; clients shown as "client 01…")
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`
- [x] My deployed paper has **all 9 sections** — Abstract at the top, Acknowledgments & data credit (https://flyrank.ai) at the bottom
- [x] **ML-12 done in this notebook's closing cells**